# Practice 104 — Regression Discontinuity Design

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_dataset, load_lee2008
from src.plotting import rdd_scatter_plot, bandwidth_sensitivity_plot, mccrary_density_plot

## Phase 1 — Sharp RDD: the local linear estimator

We simulate data with a **known** true jump `tau` (see `src/datasets.py`) and a
cutoff at 0. A sharp RDD estimate compares two *extrapolations to the cutoff* —
one local linear fit using only points just below it, one using only points just
above — instead of a naive difference in raw means.

In [ ]:
data = load_dataset("sharp", n=2000, seed=0)
print(f"True tau: {data.tau_true}")
data.as_frame().head()

### Exercise — `src/_01_local_linear_rdd.py :: local_linear_rdd`

Open `src/_01_local_linear_rdd.py`, read the `TODO(human)` block above the
function, implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_local_linear_rdd import compare_to_rdrobust, local_linear_rdd

BANDWIDTH = 0.3
compare_to_rdrobust(data.running, data.outcome, data.cutoff, bandwidth=BANDWIDTH)
fit = local_linear_rdd(data.running, data.outcome, data.cutoff, bandwidth=BANDWIDTH)

## Phase 2 — Binned means for the canonical RDD plot

A raw scatter of 2000 points hides the jump in noise. Binning the running
variable into evenly spaced, cutoff-anchored bins and plotting the mean outcome
per bin is the standard visualization (`rdrobust`'s own `rdplot` does the same
thing) — it never touches the estimate itself, only what you can see.

### Exercise — `src/_02_binned_means.py :: binned_means`

Open `src/_02_binned_means.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_binned_means import binned_means

centers, means = binned_means(data.running, data.outcome, data.cutoff, bin_width=0.05)

grid_left = np.linspace(data.running.min(), data.cutoff, 50) - data.cutoff
grid_right = np.linspace(data.cutoff, data.running.max(), 50) - data.cutoff
fig = rdd_scatter_plot(
    data.running, data.outcome, data.cutoff,
    bin_centers=centers, bin_means=means,
    fit_left=(grid_left + data.cutoff, fit.predict_left(grid_left)),
    fit_right=(grid_right + data.cutoff, fit.predict_right(grid_right)),
    title="Phase 2 — the canonical RDD figure (synthetic sharp scenario)",
)
fig

## Phase 3 — Bandwidth selection and the bias-variance tradeoff

A wider bandwidth uses more data (lower variance) but averages in more curvature
a *linear* fit can't represent (higher bias). Imbens-Kalyanaraman (2012) and
Calonico-Cattaneo-Titiunik (2014) formalize the optimal tradeoff; this phase
implements a simplified plug-in version of that idea.

### Exercise — `src/_03_bandwidth_selection.py :: rule_of_thumb_bandwidth`

Open `src/_03_bandwidth_selection.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._03_bandwidth_selection import rule_of_thumb_bandwidth

h_rot = rule_of_thumb_bandwidth(data.running, data.outcome, data.cutoff)
print(f"ROT bandwidth: {h_rot:.4f}")

bandwidth_grid = np.linspace(0.1, 0.8, 15)
estimates = [local_linear_rdd(data.running, data.outcome, data.cutoff, bandwidth=h).tau_hat for h in bandwidth_grid]
fig = bandwidth_sensitivity_plot(bandwidth_grid, estimates, selected_bandwidth=h_rot, true_value=data.tau_true)
fig

## Phase 4 — Manipulation and balance checks

Sharp/fuzzy RDD identification rests on one untestable assumption: units cannot
precisely manipulate the running variable to land on their preferred side of the
cutoff. Two falsifiable checks: the McCrary (2008) density test (does the running
variable's *density* jump at the cutoff?) and a covariate-balance placebo check
(does a pre-treatment covariate jump at the cutoff, when it shouldn't?). Both are
fully scaffolded in `src/_04_diagnostics.py` — the point of this phase is running
and interpreting them, since Phase 1's estimator already IS the balance-check
machinery, applied to a different "outcome".

In [ ]:
from src._04_diagnostics import covariate_balance_check, mccrary_density_test

lee = load_lee2008()
manipulated = load_dataset("manipulated", n=2000, seed=0)

density_lee = mccrary_density_test(lee.running, lee.cutoff)
density_manip = mccrary_density_test(manipulated.running, manipulated.cutoff)
print(f"Lee (2008), real data:    jump={density_lee.jump:+.3f}  p={density_lee.p_value:.4f}")
print(f"synthetic, manipulated:   jump={density_manip.jump:+.3f}  p={density_manip.p_value:.4f}")

fig = mccrary_density_plot(
    density_lee.bin_centers, density_lee.bin_density, lee.cutoff,
    fit_left=density_lee.fit_left, fit_right=density_lee.fit_right,
    title="McCrary density test — Lee (2008), real data",
)
fig

In [ ]:
balance = covariate_balance_check(data.running, data.covariate, data.cutoff, bandwidth=BANDWIDTH)
print(f"Covariate balance check (should be ~0): tau_hat = {balance.tau_hat:+.3f}")

## Phase 5 — Fuzzy RDD: the Wald ratio

In the `fuzzy` scenario, crossing the cutoff only shifts the *probability* of
treatment — it doesn't determine it. The reduced-form jump in Y at the cutoff
understates the true effect; dividing by the first-stage jump in treatment
take-up rescales it back, exactly like an IV/Wald estimator.

### Exercise — `src/_05_fuzzy_rdd.py :: fuzzy_rdd_wald`

Open `src/_05_fuzzy_rdd.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._05_fuzzy_rdd import fuzzy_rdd_wald

fuzzy_data = load_dataset("fuzzy", n=3000, seed=0)
fuzzy_fit = fuzzy_rdd_wald(fuzzy_data.running, fuzzy_data.outcome, fuzzy_data.treatment, fuzzy_data.cutoff, bandwidth=0.4)
print(f"first stage:    {fuzzy_fit.first_stage:.3f}")
print(f"reduced form:    {fuzzy_fit.reduced_form:.3f}")
print(f"Wald estimate:   {fuzzy_fit.tau_hat:.3f}  (true tau: {fuzzy_data.tau_true})")

## Phase 6 — End-to-end: the real Lee (2008) incumbency-advantage estimate

Everything above ran on synthetic data with a known answer. Here we run the same
pipeline — ROT bandwidth, local linear estimate, `rdrobust` cross-check, canonical
plot — on the real Lee (2008) US House elections data, where the "true" effect
isn't known but a published estimate is: Lee (2008) and Imbens & Kalyanaraman
(2012) both put the Democratic incumbency advantage at roughly 0.07-0.09
(7-9 percentage points of vote share).

In [ ]:
h_lee = rule_of_thumb_bandwidth(lee.running, lee.outcome, lee.cutoff)
print(f"ROT bandwidth (Lee data): {h_lee:.4f}")

lee_fit = local_linear_rdd(lee.running, lee.outcome, lee.cutoff, bandwidth=h_lee)
print(f"Incumbency-advantage estimate: {lee_fit.tau_hat:.4f}")

compare_to_rdrobust(lee.running, lee.outcome, lee.cutoff, bandwidth=h_lee)

lee_centers, lee_means = binned_means(lee.running, lee.outcome, lee.cutoff, bin_width=0.02)
lee_grid_left = np.linspace(lee.running.min(), lee.cutoff, 50) - lee.cutoff
lee_grid_right = np.linspace(lee.cutoff, lee.running.max(), 50) - lee.cutoff
fig = rdd_scatter_plot(
    lee.running, lee.outcome, lee.cutoff,
    bin_centers=lee_centers, bin_means=lee_means,
    fit_left=(lee_grid_left + lee.cutoff, lee_fit.predict_left(lee_grid_left)),
    fit_right=(lee_grid_right + lee.cutoff, lee_fit.predict_right(lee_grid_right)),
    title="Phase 6 — Lee (2008): Democratic incumbency advantage",
)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert abs(fit.tau_hat - data.tau_true) < 1.0, "sharp RDD estimate should be close to the true jump"
assert len(centers) == len(means) and len(centers) > 0
assert h_rot > 0
assert density_manip.p_value < density_lee.p_value, "the manipulated scenario should look less continuous than Lee (2008)"
assert abs(balance.tau_hat) < abs(fit.tau_hat), "the covariate placebo jump should be small relative to the real effect"
assert abs(fuzzy_fit.tau_hat - fuzzy_data.tau_true) < 1.5, "the Wald estimate should recover the true effect"
assert 0.0 < lee_fit.tau_hat < 0.3, "the Lee (2008) incumbency-advantage estimate should be a modest positive vote-share jump"
print("OK")